In [ ]:
import struct
from pathlib import Path
from typing import Dict

import numpy as np
from PIL import Image

from jpeg_compression_detection.paths import DATA_DIR, GENERATED_DATA_DIR, display_path


def read_jpeg_parameters(image_path: str | Path) -> Dict | None:
    """Read JPEG quantization tables and selected encoding parameters."""
    image_path = Path(image_path)
    if not image_path.is_file():
        print(f"Skipping missing JPEG file: {display_path(image_path)}")
        return None

    try:
        with Image.open(image_path) as image:
            parameters = {
                "filepath": str(image_path),
                "image_size": (image.height, image.width),
                "color_space": image.mode,
                "quant_tables": {},
                "subsampling": getattr(image, "layer", None),
                "progressive": False,
            }

        data = image_path.read_bytes()

        # Locate DQT (Define Quantization Table) markers: 0xFFDB.
        position = 0
        while position < len(data) - 1:
            if data[position] == 0xFF and data[position + 1] == 0xDB:
                position += 2
                _segment_length = struct.unpack(">H", data[position : position + 2])[0]
                position += 2

                precision_and_id = data[position]
                precision = (precision_and_id >> 4) & 0x0F
                table_id = precision_and_id & 0x0F
                position += 1

                if precision == 0:
                    table_size = 64
                    table = struct.unpack("B" * table_size, data[position : position + table_size])
                else:
                    table_size = 128
                    table = struct.unpack(">" + "H" * 64, data[position : position + table_size])

                parameters["quant_tables"][f"table_{table_id}"] = np.array(table).reshape(8, 8)
                position += table_size
            else:
                position += 1

        # SOF2 identifies progressive DCT encoding.
        position = 0
        while position < len(data) - 1:
            if data[position] == 0xFF and data[position + 1] == 0xC2:
                parameters["progressive"] = True
                break
            position += 1

        return parameters
    except Exception as exc:
        print(f"Unable to read {display_path(image_path)}: {exc}")
        return None


def print_jpeg_info(parameters: Dict | None) -> None:
    """Print JPEG parameters in a compact, readable form."""
    if parameters is None:
        print("No JPEG parameters to display")
        return

    print(f"\n{'=' * 60}")
    print(f"JPEG file: {display_path(parameters['filepath'])}")
    print(f"{'=' * 60}")
    print(f"Image size: {parameters['image_size'][0]} x {parameters['image_size'][1]} px")
    print(f"Color space: {parameters['color_space']}")
    print(f"Progressive JPEG: {'yes' if parameters['progressive'] else 'no'}")
    if parameters["subsampling"]:
        print(f"Subsampling: {parameters['subsampling']}")

    print(f"\nQuantization tables found: {len(parameters['quant_tables'])}")
    for table_name, table in parameters["quant_tables"].items():
        print(f"\n{table_name}:")
        print(table)


example_image_path = DATA_DIR / "test-images" / "2-min.jpg"
example_parameters = read_jpeg_parameters(example_image_path)
if example_parameters is not None:
    print_jpeg_info(example_parameters)


# Repeated compression at the same JPEG quality


In [ ]:
same_quality_dir = GENERATED_DATA_DIR / "multiple-compression-same-quality"
same_quality_files = [
    same_quality_dir / "01 (1)_q70.jpg",
    same_quality_dir / "01 (1)_q70.jpg.q70.jpg",
    same_quality_dir / "01 (1)_q70.jpg.q70.q70.jpg",
]

for image_path in same_quality_files:
    parameters = read_jpeg_parameters(image_path)
    if parameters is not None:
        print_jpeg_info(parameters)


In [ ]:
# Compare the quantization tables after successive recompressions.
parameter_sets = [read_jpeg_parameters(path) for path in same_quality_files]

if all(parameters is not None for parameters in parameter_sets):
    first, second, third = parameter_sets
    print("\n" + "=" * 60)
    print("Quantization-table comparison")
    print("=" * 60)

    for table_name in first["quant_tables"]:
        print(f"\n{table_name}:")
        print("\nOne compression (q70):")
        print(first["quant_tables"][table_name])
        print("\nTwo compressions (q70):")
        print(second["quant_tables"][table_name])
        print("\nThree compressions (q70):")
        print(third["quant_tables"][table_name])

        difference_1_2 = second["quant_tables"][table_name] - first["quant_tables"][table_name]
        difference_2_3 = third["quant_tables"][table_name] - second["quant_tables"][table_name]
        print("\nDifference (two minus one):")
        print(difference_1_2)
        print("\nDifference (three minus two):")
        print(difference_2_3)
else:
    print("Skipping comparison because one or more input JPEG files are missing.")


# In this small example, repeated compression at the same quality does not change the quantization tables.


In [ ]:
def read_huffman_tables(image_path: str | Path) -> Dict | None:
    """Read Huffman tables from a JPEG file."""
    image_path = Path(image_path)
    if not image_path.is_file():
        print(f"Skipping missing JPEG file: {display_path(image_path)}")
        return None

    huffman_tables = {"DC": {}, "AC": {}}
    try:
        data = image_path.read_bytes()
        position = 0
        while position < len(data) - 1:
            if data[position] == 0xFF and data[position + 1] == 0xC4:
                position += 2
                segment_length = struct.unpack(">H", data[position : position + 2])[0]
                position += 2
                end_position = position + segment_length - 2

                while position < end_position:
                    table_info = data[position]
                    table_class = (table_info >> 4) & 0x0F
                    table_id = table_info & 0x0F
                    position += 1

                    code_counts = list(data[position : position + 16])
                    position += 16
                    total_codes = sum(code_counts)
                    symbols = list(data[position : position + total_codes])
                    position += total_codes

                    table_type = "DC" if table_class == 0 else "AC"
                    huffman_tables[table_type][f"table_{table_id}"] = {
                        "num_codes": code_counts,
                        "symbols": symbols,
                        "total_codes": total_codes,
                    }
            else:
                position += 1
        return huffman_tables
    except Exception as exc:
        print(f"Unable to read Huffman tables from {display_path(image_path)}: {exc}")
        return None


def print_huffman_info(huffman_tables: Dict | None, image_path: str | Path = "") -> None:
    """Print a readable summary of JPEG Huffman tables."""
    if huffman_tables is None:
        print("No Huffman-table data to display")
        return

    print(f"\n{'=' * 60}")
    print(f"Huffman tables: {display_path(image_path)}")
    print(f"{'=' * 60}")
    for table_type in ("DC", "AC"):
        print(f"\n{table_type} tables:")
        for table_name, table_data in huffman_tables[table_type].items():
            print(f"\n  {table_name}:")
            print(f"    Code counts by length: {table_data['num_codes']}")
            print(f"    Total number of codes: {table_data['total_codes']}")
            print(f"    First 20 symbols: {table_data['symbols'][:20]}")


for image_path in same_quality_files[:2]:
    print_huffman_info(read_huffman_tables(image_path), image_path.name)


In JPEG Huffman coding, DC and AC refer to the same coefficient classes as in the DCT, but they use separate Huffman tables because their statistical properties differ.

DC Huffman tables encode the DC coefficient, which represents the average value of an 8 x 8 block. JPEG usually encodes the difference between the current and previous block's DC coefficients (DPCM). These differences are generally small because neighbouring blocks tend to have similar brightness. A DC table therefore usually contains only 12–16 symbols.

AC Huffman tables encode the remaining 63 frequency coefficients. They encode run-length/value pairs: the number of consecutive zeros followed by a coefficient value. Many high-frequency AC coefficients are zero, so AC tables are larger and commonly contain 162–256 symbols.

The code stores them as:

```python
huffman_tables = {
    "DC": {},  # Tables for DC coefficients.
    "AC": {},  # Tables for AC coefficients.
}
```

A typical JPEG contains four Huffman tables: DC and AC table 0 for luminance (Y), and DC and AC table 1 for chrominance (Cb and Cr).
